# z615 - DTW + Clustering + LightGBM por cluster (product_id)
780 series -- un solo lote, matriz de distancias completa (~304k pares, trivial). Escala -> DTW -> clustering jerarquico -> un LightGBM por cluster -> submit combinado.

In [1]:
!pip install -q dtaidistance scipy scikit-learn lightgbm pyarrow polars

In [2]:
import os
import numpy as np
import polars as pl
import lightgbm as lgb
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'LGB10_DTW_PROD',
    'kaggle_competition': 'labo-iii-2026-ba',
    'features_path': '/home/ds/exp/FE609/tb_features_FE609.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'horizonte_meses': 2,
    'periodo_ultimo_dato': 201912,
    'semilla': 102103,
    'n_clusters': 10,
    'min_filas_por_cluster': 200
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB10_DTW_PROD


## 1. Armar la matriz producto x periodo (serie cruda de tn, escalada por producto)

In [4]:
df = pl.read_parquet(PARAM['features_path'])

wide = df.select(["product_id", "periodo", "tn"]).pivot(
    values="tn", index="product_id", columns="periodo"
).sort("product_id")

product_ids = wide["product_id"].to_numpy()
matriz = wide.drop("product_id").to_numpy().astype(np.float64)
matriz = np.nan_to_num(matriz, nan=0.0)  # periodos fuera de vida del producto -> 0
print(matriz.shape)

(1233, 36)


In [5]:
# escalado z-score por serie (DTW mide FORMA, no magnitud -- sin esto, agrupa por volumen)
medias = matriz.mean(axis=1, keepdims=True)
stds = matriz.std(axis=1, keepdims=True)
stds[stds == 0] = 1.0
matriz_escalada = (matriz - medias) / stds

## 2. Matriz de distancias DTW + clustering jerarquico

In [9]:
print("infinitos en dist_matrix:", np.sum(~np.isfinite(dist_matrix)))
print("nans en matriz_escalada:", np.sum(~np.isfinite(matriz_escalada)))

filas_malas = np.where(~np.isfinite(matriz_escalada).all(axis=1))[0]
print("filas de matriz_escalada con algun no-finito:", filas_malas)

infinitos en dist_matrix: 20
nans en matriz_escalada: 0
filas de matriz_escalada con algun no-finito: []


In [10]:
series_lista = [matriz_escalada[i] for i in range(matriz_escalada.shape[0])]
dist_matrix = dtw.distance_matrix_fast(series_lista)

dist_matrix = np.where(np.isfinite(dist_matrix), dist_matrix, dist_matrix.T)
np.fill_diagonal(dist_matrix, 0.0)

# los pares que sigan infinitos (rarezas del calculo, ~20 de 304k) -> distancia maxima finita
max_finita = np.max(dist_matrix[np.isfinite(dist_matrix)])
dist_matrix = np.where(np.isfinite(dist_matrix), dist_matrix, max_finita)
np.fill_diagonal(dist_matrix, 0.0)

print("infinitos restantes:", np.sum(~np.isfinite(dist_matrix)))

dist_condensada = squareform(dist_matrix, checks=False)
Z = linkage(dist_condensada, method="average")
clusters = fcluster(Z, t=PARAM['n_clusters'], criterion="maxclust")

tb_clusters = pl.DataFrame({"product_id": product_ids, "cluster_id": clusters})
print(tb_clusters.group_by("cluster_id").len().sort("cluster_id"))

infinitos restantes: 0
shape: (10, 2)
┌────────────┬─────┐
│ cluster_id ┆ len │
│ ---        ┆ --- │
│ i32        ┆ u32 │
╞════════════╪═════╡
│ 1          ┆ 3   │
│ 2          ┆ 13  │
│ 3          ┆ 17  │
│ 4          ┆ 216 │
│ 5          ┆ 507 │
│ 6          ┆ 113 │
│ 7          ┆ 9   │
│ 8          ┆ 3   │
│ 9          ┆ 260 │
│ 10         ┆ 92  │
└────────────┴─────┘


In [11]:
df = df.join(tb_clusters, on="product_id", how="left")
df.write_parquet(os.path.join(ruta, "tb_features_con_cluster.parquet"))
print(os.path.join(ruta, "tb_features_con_cluster.parquet"))

/home/ds/exp/LGB10_DTW_PROD/tb_features_con_cluster.parquet


## 3. Un LightGBM por cluster
Hiperparametros fijos (no Optuna) para mantener el tiempo total razonable con 10 entrenamientos. Clusters con menos de `min_filas_por_cluster` se entrenan igual, pero se marcan para revisar (poca data).

In [12]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

H = PARAM['horizonte_meses']
df = df.sort(["product_id", "periodo"])
df = df.with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("tn_target")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

m_201910 = periodo_a_meses(201910)
m_201911 = periodo_a_meses(201911)
m_201912 = periodo_a_meses(201912)

cols_excluir = {"tn", "tn_target", "tn_shift1", "periodo", "periodo_target_m", "nacimiento_m", "cluster_id"}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]

params_fijos = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'seed': PARAM['semilla']
}

def a_pandas(tabla):
    pdf = tabla.select(features + ["tn_target"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

In [13]:
predicciones_totales = []

for cluster_id in sorted(df["cluster_id"].unique().to_list()):
    sub = df.filter(pl.col("cluster_id") == cluster_id)
    sub_valido = sub.filter(pl.col("tn_target").is_not_null())

    train = sub_valido.filter(pl.col("periodo_target_m") <= m_201910)
    valid = sub_valido.filter(
        (pl.col("periodo_target_m") >= m_201911) & (pl.col("periodo_target_m") <= m_201912)
    )

    if train.height < 50 or valid.height < 5:
        print(f"cluster {cluster_id}: muy poca data (train={train.height}, valid={valid.height}), se salta")
        continue

    train_pd = a_pandas(train)
    valid_pd = a_pandas(valid)

    X_train = train_pd[features]
    y_train = np.log1p(train_pd["tn_target"].clip(lower=0))
    X_valid = valid_pd[features]
    y_valid = np.log1p(valid_pd["tn_target"].clip(lower=0))

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                          params={'feature_pre_filter': False})
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                          params={'feature_pre_filter': False})

    modelo = lgb.train(
        params_fijos, dtrain, num_boost_round=2000,
        valid_sets=[dvalid], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    futuro = sub.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
    futuro_pd = futuro.select(features).to_pandas()
    for c in categoricas:
        futuro_pd[c] = futuro_pd[c].astype("category")

    pred_log = modelo.predict(futuro_pd, num_iteration=modelo.best_iteration)
    pred_tn = np.clip(np.expm1(pred_log), 0, None)

    res = futuro.select(["product_id"]).to_pandas()
    res["tn"] = pred_tn
    predicciones_totales.append(res)

    print(f"cluster {cluster_id}: train={train.height} valid={valid.height} mejor_iter={modelo.best_iteration}")

cluster 1: muy poca data (train=0, valid=0), se salta
cluster 2: muy poca data (train=35, valid=0), se salta
cluster 3: train=544 valid=34 mejor_iter=110
cluster 4: train=5810 valid=356 mejor_iter=62
cluster 5: train=15646 valid=797 mejor_iter=106
cluster 6: train=1935 valid=54 mejor_iter=82
cluster 7: train=288 valid=18 mejor_iter=50
cluster 8: train=96 valid=6 mejor_iter=106
cluster 9: train=2351 valid=478 mejor_iter=91
cluster 10: train=544 valid=84 mejor_iter=46


## 4. Combinar predicciones de todos los clusters y armar submit

In [14]:
import pandas as pd
resultado = pd.concat(predicciones_totales, ignore_index=True)

apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()
submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos (productos sin cluster valido, revisar):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos (productos sin cluster valido, revisar): 0
/home/ds/exp/LGB10_DTW_PROD/LGB10_DTW_PROD_submit.csv


,product_id,tn
0,20001,1417.306235
1,20002,1225.145345
2,20003,721.063543
3,20004,514.830109
4,20005,509.593673


In [15]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} DTW clusters producto")

100%|██████████| 18.7k/18.7k [00:00<00:00, 59.2kB/s]


99 submissions remaining today.
Successfully submitted to Labo III, 2026 BA